## Transformación y limpieza de datos

### Objetivo
Este notebook aplica las transformaciones necesarias sobre los datos extraídos para prepararlos para su carga en la base de datos analítica.

Las operaciones incluyen:
- **Tipado de columnas**: conversión de IDs numéricos a cadena (`str`) para tratarlos como variables categóricas.
- **Mapeo de territorios**: asignación del `id_territorio` desde la tabla dimensión a los DataFrames de hechos (constituidas y disueltas), y normalización de nombres (minúsculas, guiones bajos).
- **Normalización de texto**: estandarización de nombres de sectores, meses, razones de disolución y tipos de medida.
- **Eliminación de filas totales**: filtrado de filas "Mercantiles" que agregan información ya presente en los desgloses por tipo societario.
- **Abreviaturas**: conversión de tipos societarios a siglas (S.A., S.L., S.Com./S.C.).

### Metodología
1. **Carga** de los CSV desde `../files/data_raw/`.
2. **Transformaciones** aplicadas mediante funciones del módulo `src.transformation` y mapeos manuales.
3. **Exportación** de los datasets procesados a `../files/data_processed/` para su consumo en la fase de carga.

In [1]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))

# Importación del módulo de transformación
from src.transformation import trans_str
from src.transformation import trans_normal

# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)

## Paso 1: Carga de datos

In [2]:
df_empr_const = pd.read_csv('../files/data_raw/empresas_constituidas.csv')
df_empr_dis = pd.read_csv('../files/data_raw/empresas_disueltas.csv')
df_ipc = pd.read_csv('../files/data_raw/ipc.csv')
df_sectores_ipc = pd.read_csv('../files/data_raw/sectores_ipc.csv')
df_territorio = pd.read_csv('../files/data_raw/territorio.csv')
df_tiempo = pd.read_csv('../files/data_raw/tiempo.csv')
df_tipo_medida = pd.read_csv('../files/data_raw/tipo_medida.csv')

## Paso 2: Conversión de tipos
Se convierten las columnas de IDs y períodos de numéricas a string, ya que actúan como variables categóricas y no requieren operaciones aritméticas.

In [3]:
# Listas de columnas a transformar de cada DataFrame
lista_const = ['id_const', 'id_tiempo']
lista_dis = ['id_dis', 'id_tiempo']
lista_ipc = ['id_tiempo', 'id_territorio', 'id_sector', 'id_medida']
lista_sector_ipc = ['id_sector']
lista_territorio = ['id_territorio']
lista_tiempo = ['id_tiempo', 'anio', 'mes']
lista_tipo_medida = ['id_medida']

In [4]:
trans_str.int_a_str(df_empr_const, lista_const)
trans_str.int_a_str(df_empr_dis, lista_dis)
trans_str.int_a_str(df_ipc, lista_ipc)
trans_str.int_a_str(df_sectores_ipc, lista_sector_ipc)
trans_str.int_a_str(df_territorio, lista_territorio)
trans_str.int_a_str(df_tiempo, lista_tiempo)
trans_str.int_a_str(df_tipo_medida, lista_tipo_medida)

,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [5]:
df_empr_const.info()

<class 'pandas.DataFrame'>
RangeIndex: 16720 entries, 0 to 16719
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_const           16720 non-null  str  
 1   territorio         16720 non-null  str  
 2   id_tiempo          16720 non-null  str  
 3   tipo               16720 non-null  str  
 4   numero_sociedades  16720 non-null  int64
 5   capital            16720 non-null  int64
dtypes: int64(2), str(4)
memory usage: 783.9 KB


In [6]:
df_empr_dis.info()

<class 'pandas.DataFrame'>
RangeIndex: 12540 entries, 0 to 12539
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_dis             12540 non-null  str  
 1   territorio         12540 non-null  str  
 2   id_tiempo          12540 non-null  str  
 3   razon              12540 non-null  str  
 4   numero_sociedades  12540 non-null  int64
dtypes: int64(1), str(4)
memory usage: 490.0 KB


In [7]:
df_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 328162 entries, 0 to 328161
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id_tiempo      328162 non-null  str    
 1   id_territorio  328162 non-null  str    
 2   id_sector      328162 non-null  str    
 3   id_medida      328162 non-null  str    
 4   valor_ipc      328162 non-null  float64
dtypes: float64(1), str(4)
memory usage: 12.5 MB


In [8]:
df_sectores_ipc.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_sector      14 non-null     str  
 1   nombre_sector  14 non-null     str  
dtypes: str(2)
memory usage: 356.0 bytes


In [9]:
df_territorio.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_territorio      20 non-null     str  
 1   nombre_territorio  20 non-null     str  
dtypes: str(2)
memory usage: 452.0 bytes


In [10]:
df_territorio.head()

,id_territorio,nombre_territorio
0,1,Nacional
1,2,Andalucía
2,3,Aragón
3,4,"Asturias, Principado de"
4,5,"Balears, Illes"


In [11]:
df_tiempo.info()

<class 'pandas.DataFrame'>
RangeIndex: 294 entries, 0 to 293
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_tiempo   294 non-null    str  
 1   anio        294 non-null    str  
 2   mes         294 non-null    str  
 3   nombre_mes  294 non-null    str  
dtypes: str(4)
memory usage: 9.3 KB


In [12]:
df_tipo_medida.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_medida      4 non-null      str  
 1   nombre_medida  4 non-null      str  
dtypes: str(2)
memory usage: 196.0 bytes


## Paso 3: Normalización de territorios
Se asigna `id_territorio` a los DataFrames de hechos (constituidas y disueltas) mediante mapeo con la tabla dimensión.  
Se eliminan las columnas de texto de territorio de ambos DataFrames ya que quedan representadas por `id_territorio`.  
Se normalizan los nombres de territorio: minúsculas, sin acentos, separados por guiones bajos.

In [13]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
mapeo_territorios_nombre = {
    "Nacional": "nacional",
    "Andalucía": "andalucia",
    "Aragón": "aragon",
    "Asturias, Principado de": "principado_de_asturias",
    "Balears, Illes": "islas_baleares",
    "Canarias": "canarias",
    "Cantabria": "cantabria",
    "Castilla y León": "castilla_y_leon",
    "Castilla - La Mancha": "castilla_la_mancha",
    "Cataluña": "cataluna",
    "Comunitat Valenciana": "comunidad_valenciana",
    "Extremadura": "extremadura",
    "Galicia": "galicia",
    "Madrid, Comunidad de": "comunidad_de_madrid",
    "Murcia, Región de": "region_de_murcia",
    "Navarra, Comunidad Foral de": "comunidad_foral_de_navarra",
    "País Vasco": "pais_vasco",
    "Rioja, La": "la_rioja",
    "Ceuta": "ceuta",
    "Melilla": "melilla"
}

In [14]:
# Diccionario territorio -> id_territorio, usando el propio df_territorio
mapeo_territorios = dict(zip(df_territorio['nombre_territorio'], df_territorio['id_territorio']))

# Aplicar el mapeo
df_empr_const['id_territorio'] = df_empr_const['territorio'].map(mapeo_territorios)
df_empr_dis['id_territorio'] = df_empr_dis['territorio'].map(mapeo_territorios)

In [15]:
df_empr_const.sample(10)

,id_const,territorio,id_tiempo,tipo,numero_sociedades,capital,id_territorio
3654,3655,"Rioja, La",201502,Mercantiles,30,482000,18
958,959,Canarias,201910,Mercantiles,288,6264000,6
6447,6448,Extremadura,202009,Sociedades anónimas,0,0,12
12567,12568,Andalucía,202401,S. Comanditarias y S. Colectivas,1,0,2
4847,4848,"Balears, Illes",202509,Sociedades anónimas,0,0,5
8155,8156,Melilla,202501,Sociedades anónimas,0,0,20
11068,11069,"Madrid, Comunidad de",202008,Sociedades de responsabilidad limitada,1211,37365000,14
9236,9237,"Balears, Illes",200804,Sociedades de responsabilidad limitada,297,5993000,5
8887,8888,"Asturias, Principado de",201901,Sociedades de responsabilidad limitada,91,2893000,4
1651,1652,Castilla - La Mancha,201701,Mercantiles,255,8495000,9


In [16]:
df_empr_dis.sample(10)

,id_dis,territorio,id_tiempo,razon,numero_sociedades,id_territorio
12518,12519,Melilla,200910,Otras,0,20
7573,7574,País Vasco,201807,Por fusión,9,17
12471,12472,Melilla,201309,Otras,0,20
10588,10589,Extremadura,202312,Otras,2,12
8146,8147,Melilla,202510,Por fusión,0,20
11230,11231,"Murcia, Región de",202506,Otras,5,15
1532,1533,Castilla y León,200808,Voluntaria,23,8
495,496,"Asturias, Principado de",202109,Voluntaria,22,4
645,646,"Asturias, Principado de",200903,Voluntaria,30,4
1862,1863,Cataluña,201710,Voluntaria,65,10


In [17]:
df_empr_dis.drop(columns=["territorio"],inplace=True)

In [18]:
df_empr_const.drop(columns=["territorio"],inplace=True)

In [19]:
df_territorio['nombre_territorio'] = df_territorio['nombre_territorio'].replace(mapeo_territorios_nombre)

In [20]:
df_territorio['nombre_territorio'].unique()

<StringArray>
[                  'nacional',                  'andalucia',
                     'aragon',     'principado_de_asturias',
             'islas_baleares',                   'canarias',
                  'cantabria',            'castilla_y_leon',
         'castilla_la_mancha',                   'cataluna',
       'comunidad_valenciana',                'extremadura',
                    'galicia',        'comunidad_de_madrid',
           'region_de_murcia', 'comunidad_foral_de_navarra',
                 'pais_vasco',                   'la_rioja',
                      'ceuta',                    'melilla']
Length: 20, dtype: str

## Paso 4: Normalización de texto
Se aplica minúsculas y guiones bajos a las columnas de texto de las tablas dimensionales: `razon` (disueltas), `nombre_sector` (sectores IPC), `nombre_mes` (tiempo) y `nombre_medida` (tipo_medida).

In [21]:
# Así se aplica una función a los DATOS de una columna
df_empr_dis["razon"] = df_empr_dis["razon"].apply(trans_normal.normalizar_col)
df_sectores_ipc["nombre_sector"] = df_sectores_ipc["nombre_sector"].apply(trans_normal.normalizar_col)
df_tiempo["nombre_mes"] = df_tiempo["nombre_mes"].apply(trans_normal.normalizar_col)
df_tipo_medida["nombre_medida"] = df_tipo_medida["nombre_medida"].apply(trans_normal.normalizar_col)

In [22]:
df_empr_dis.sample(10)

,id_dis,id_tiempo,razon,numero_sociedades,id_territorio
9693,9694,202503,otras,6,8
488,489,202204,voluntaria,24,4
7565,7566,201903,por_fusion,3,17
5216,5217,201304,por_fusion,10,6
4082,4083,201602,voluntaria,1,20
4032,4033,202004,voluntaria,0,20
5923,5924,200905,por_fusion,2,9
997,998,201607,voluntaria,33,6
3184,3185,201708,voluntaria,1,16
2317,2318,201607,voluntaria,15,12


In [23]:
df_sectores_ipc.sample(10)

,id_sector,nombre_sector
0,1,indice_general
8,9,informacion_y_comunicaciones
5,6,muebles_articulos_del_hogar_y_articulos_para_e...
10,11,ensenanza
13,14,cuidado_personal_proteccion_social_y_bienes_y_...
3,4,vestido_y_calzado
9,10,actividades_recreativas_deporte_y_cultura
12,13,seguros_y_servicios_financieros
4,5,vivienda_agua_electricidad_gas_y_otros_combust...
7,8,transporte


In [24]:
df_tiempo.sample(10)

,id_tiempo,anio,mes,nombre_mes
215,200806,2008,6,junio
14,202503,2025,3,marzo
95,201806,2018,6,junio
248,200509,2005,9,septiembre
267,200402,2004,2,febrero
96,201805,2018,5,mayo
82,201907,2019,7,julio
27,202402,2024,2,febrero
151,201310,2013,10,octubre
114,201611,2016,11,noviembre


In [25]:
df_tipo_medida.sample(4)

,id_medida,nombre_medida
3,4,variacion_en_lo_que_va_de_ano
1,2,variacion_mensual
0,1,indice
2,3,variacion_anual


## Paso 5: Eliminación de filas agregadas
Se filtran las filas con `tipo = "Mercantiles"` de `empresas_constituidas`, ya que representan la suma de S.A. y S.L. y generarían datos duplicados en el análisis.

In [26]:
# Eliminamos las filas que contienen "Mercantiles"
df_empr_const = df_empr_const[df_empr_const['tipo'] != 'Mercantiles']

In [27]:
df_empr_const['tipo'].unique()

<StringArray>
[                   'Sociedades anónimas',
 'Sociedades de responsabilidad limitada',
       'S. Comanditarias y S. Colectivas']
Length: 3, dtype: str

In [28]:
df_empr_const.shape

(12540, 6)

## Paso 6: Abreviatura de tipos societarios
Se sustituyen los nombres completos de los tipos societarios por sus siglas: S.A., S.L. y S.Com./S.C.

In [29]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
dicc_siglas = {
    'Sociedades de responsabilidad limitada': 'S.L.',
    'Sociedades anónimas': 'S.A.',
    'S. Comanditarias y S. Colectivas': 'S.Com./S.C.'
}

In [30]:
# Aplicamos el cambio a la columna 'tipo'
df_empr_const['tipo'] = df_empr_const['tipo'].replace(dicc_siglas)

In [31]:
df_empr_const['tipo'].unique()

<StringArray>
['S.A.', 'S.L.', 'S.Com./S.C.']
Length: 3, dtype: str

## Paso 7: Exportación de datos procesados
Se guardan los 7 DataFrames transformados en `../files/data_processed/` para su consumo en la fase de carga.

In [32]:
df_empr_const.to_csv('../files/data_processed/empresas_constituidas.csv', index=False)
df_empr_dis.to_csv('../files/data_processed/empresas_disueltas.csv', index=False)
df_ipc.to_csv('../files/data_processed/ipc.csv', index=False)
df_sectores_ipc.to_csv('../files/data_processed/sectores_ipc.csv', index=False)
df_territorio.to_csv('../files/data_processed/territorio.csv', index=False)
df_tiempo.to_csv('../files/data_processed/tiempo.csv', index=False)
df_tipo_medida.to_csv('../files/data_processed/tipo_medida.csv', index=False)

In [33]:
# NOTA: Los nombres normalizados están pensados para uso en BD y código.
# Para presentación en PowerBI, se recomienda crear columnas display 
# con los nombres originales del INE usando Data Category o DAX.